# ETL Pipeline: PDDikti → Star Schema Data Warehouse
## Universitas Siliwangi — Analisis Rasio Dosen:Mahasiswa

**Notebook ini mendokumentasikan proses ETL (Extract → Transform → Load) sebagai bagian dari implementasi BAB 4.**

| Fase | Kegiatan |
|------|----------|
| Extract | Membaca data mentah hasil scraping PDDikti dari Google Drive |
| Transform | Cleaning, parsing, normalisasi format |
| Load | Menyimpan ke Star Schema (4 tabel) + flat table untuk BI |

---
**Sumber Data:** PDDikti (https://pddikti.kemdiktisaintek.go.id)  
**Periode:** Ganjil 2023 — Ganjil 2025 (5 periode)  
**Scope:** Universitas Siliwangi (semua program studi aktif)  
**Storage:** Google Drive — My Drive/College/

---
## 0. Mount Google Drive

Jalankan cell ini terlebih dahulu. Akan muncul popup untuk mengizinkan akses Google Drive.
Setelah berhasil, folder Drive akan tersedia di `/content/drive/MyDrive/`.

In [1]:
# [LOCAL MODE] - Google Drive tidak diperlukan
print('Mode: Lokal OK')


Mode: Lokal OK


---
## 1. Import Library & Definisi Path

In [2]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

PATH_RAW_PRODI  = 'd:/Code/Tugas Akhir/Data/Raw/prodi_raw.csv'
PATH_RAW_UNIV   = 'd:/Code/Tugas Akhir/Data/Raw/universitas_raw.csv'
PATH_OUT_SCHEMA = 'd:/Code/Tugas Akhir/Data/Star_Schema/'
PATH_OUT_MASTER = 'd:/Code/Tugas Akhir/Data/Processed/master_looker_unsil.csv'

os.makedirs(PATH_OUT_SCHEMA, exist_ok=True)
print('Paths OK - mode lokal aktif')


Paths OK - mode lokal aktif


---
## 2. EXTRACT — Membaca Data Mentah

Data diperoleh dari scraping PDDikti menggunakan Selenium dan telah disimpan ke Google Drive.  
Format data mentah masih berupa teks apa adanya dari website PDDikti — belum ada transformasi apapun.

In [3]:
# Baca data mentah hasil scraping dari path lokal
df_prodi_raw = pd.read_csv(PATH_RAW_PRODI)
df_univ_raw  = pd.read_csv(PATH_RAW_UNIV)

print('Data mentah berhasil dibaca.')
print('df_prodi_raw:', df_prodi_raw.shape)
print('df_univ_raw :', df_univ_raw.shape)


Data mentah berhasil dibaca.
df_prodi_raw: (16564, 16)
df_univ_raw : (44, 14)


In [4]:
# Preview data SEBELUM transformasi
print('[TABEL 1] Sample Data Mentah Prodi (5 baris pertama):')
print(df_prodi_raw.head(5))

print('\n[TABEL 2] Data Universitas Siliwangi (metadata):')
print(df_univ_raw)

print('\n[STATISTIK DATA MENTAH]')
print(f'Total baris prodi (semua periode) : {len(df_prodi_raw)}')
print(f'Periode yang tersedia             : {sorted(df_prodi_raw["tahun_pelaporan"].unique())}')
print(f'Jumlah prodi unik                 : {df_prodi_raw["nama_program_studi"].nunique()}')
print(f'\nNilai kosong per kolom:')
print(df_prodi_raw.isnull().sum()[df_prodi_raw.isnull().sum() > 0])

[TABEL 1] Sample Data Mentah Prodi (5 baris pertama):
                         nama_universitas kode_pt status_pt_univ  \
0  Institut Seni Indonesia Padang Panjang       -        Jenjang   
1  Institut Seni Indonesia Padang Panjang       -        Jenjang   
2  Institut Seni Indonesia Padang Panjang       -        Jenjang   
3  Institut Seni Indonesia Padang Panjang       -        Jenjang   
4  Institut Seni Indonesia Padang Panjang       -        Jenjang   

  akreditasi_pt_univ tahun_pelaporan  kode_prodi        nama_program_studi  \
0         1\ndari\n6     Ganjil 2024       90345                   Animasi   
1         1\ndari\n6     Ganjil 2024       82201        Antropologi Budaya   
2         1\ndari\n6     Ganjil 2024       90241  Desain Komunikasi Visual   
3         1\ndari\n6     Ganjil 2024       90331               Desain Mode   
4         1\ndari\n6     Ganjil 2024       90231             Desain Produk   

  status_prodi jenjang akreditasi_prodi  jumlah_dosen_penghitung_ras

---
## 3. TRANSFORM — Cleaning & Normalisasi

Tahap transformasi terdiri atas 6 langkah berurutan:
- **Langkah 0**: Filter scope — ambil hanya data Universitas Siliwangi
- **Langkah 1**: Hapus baris dengan data kritis yang kosong
- **Langkah 2**: Parsing kolom `tahun_pelaporan` → `semester` + `tahun`
- **Langkah 3**: Konversi kolom numerik
- **Langkah 4**: Parsing kolom `rasio_dosen_mahasiswa` → nilai numerik `nilai_rasio`
- **Langkah 5**: Standarisasi metadata universitas

> **Catatan:** Nilai "-" pada kolom `rasio_dosen_mahasiswa` di PDDikti muncul pada program studi  
> dengan `jumlah_mahasiswa = 0`. Fungsi `parse_rasio` akan menghasilkan `NaN` untuk nilai tersebut.  
> Hal ini bukan error — mencerminkan kondisi aktual data sumber.

In [5]:
# ── LANGKAH 0: Filter scope penelitian → Universitas Siliwangi ──────────────
print('=' * 60)
print('LANGKAH 0: FILTER SCOPE — Universitas Siliwangi')
print('=' * 60)

df = df_prodi_raw.copy()
total_sebelum = len(df)
univ_sebelum  = df['nama_universitas'].nunique() if 'nama_universitas' in df.columns else '?'

# Filter: hanya baris yang nama_universitas mengandung 'Siliwangi'
df = df[
    df['nama_universitas'].str.contains('Siliwangi', case=False, na=False)
].copy()

# Hardcode kode_pt resmi Unsil
df['kode_pt'] = '002008'

print(f'Data masuk  : {total_sebelum:,} baris | {univ_sebelum} universitas')
print(f'Data keluar : {len(df):,} baris | {df["nama_program_studi"].nunique()} prodi unik')
print(f'Universitas : {df["nama_universitas"].unique()}')
print(f'Periode     : {sorted(df["tahun_pelaporan"].unique())}')
print('\n[OK] Filter scope selesai — data sudah dibatasi ke Universitas Siliwangi.')

LANGKAH 0: FILTER SCOPE — Universitas Siliwangi
Data masuk  : 16,564 baris | 44 universitas
Data keluar : 200 baris | 34 prodi unik
Universitas : <StringArray>
['Universitas Siliwangi']
Length: 1, dtype: str
Periode     : ['Ganjil 2023', 'Ganjil 2024', 'Ganjil 2025', 'Genap 2023', 'Genap 2024']

[OK] Filter scope selesai — data sudah dibatasi ke Universitas Siliwangi.


In [6]:
# Langkah 1–5: Cleaning & Normalisasi

# 1. Hapus baris dengan nilai kritis kosong
before = len(df)
df = df.dropna(subset=['kode_prodi', 'tahun_pelaporan', 'rasio_dosen_mahasiswa'])
print(f'[1] Drop null kritis: {before} → {len(df)} baris (dihapus {before - len(df)} baris)')

# 2. Parsing tahun_pelaporan → semester + tahun
df[['semester', 'tahun']] = df['tahun_pelaporan'].str.split(' ', n=1, expand=True)
print(f'[2] Parsing periode: {sorted(df["tahun_pelaporan"].unique())}')

# 3. Konversi kolom numerik
num_cols = ['jumlah_dosen_penghitung_rasio', 'dosen_tetap', 'dosen_tidak_tetap',
            'total_dosen', 'jumlah_mahasiswa']
for col in num_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')
print(f'[3] Konversi numerik: {num_cols}')

# 4. Parsing rasio '1:X' → nilai numerik X
# Nilai "-" dari PDDikti (prodi dengan 0 mahasiswa) → menghasilkan NaN (bukan error)
def parse_rasio(s):
    try:
        if pd.isna(s): return np.nan
        parts = str(s).split(':')
        return float(parts[1]) if len(parts) == 2 else np.nan
    except:
        return np.nan

df['nilai_rasio'] = df['rasio_dosen_mahasiswa'].apply(parse_rasio)
n_nan = df['nilai_rasio'].isna().sum()
print(f'[4] Parse rasio: {len(df) - n_nan} baris berhasil, {n_nan} baris NaN (prodi dengan 0 mahasiswa)')

# 5. Standarisasi metadata universitas
df['nama_universitas']   = 'Universitas Siliwangi'
df['status_pt_univ']     = 'PTN'
df['akreditasi_pt_univ'] = 'Unggul'
df['kode_pt']            = '002008'
print(f'[5] Standarisasi metadata Unsil selesai.')

print(f'\n✅ Transformasi selesai: {len(df)} baris siap diproses.')

[1] Drop null kritis: 200 → 200 baris (dihapus 0 baris)
[2] Parsing periode: ['Ganjil 2023', 'Ganjil 2024', 'Ganjil 2025', 'Genap 2023', 'Genap 2024']
[3] Konversi numerik: ['jumlah_dosen_penghitung_rasio', 'dosen_tetap', 'dosen_tidak_tetap', 'total_dosen', 'jumlah_mahasiswa']
[4] Parse rasio: 175 baris berhasil, 25 baris NaN (prodi dengan 0 mahasiswa)
[5] Standarisasi metadata Unsil selesai.

✅ Transformasi selesai: 200 baris siap diproses.


In [7]:
# Preview data SETELAH transformasi
print('[TABEL 3] Sample Data Setelah Transformasi (5 baris pertama):')
cols_tampil = ['tahun_pelaporan', 'semester', 'tahun', 'nama_program_studi', 'jenjang',
               'total_dosen', 'jumlah_mahasiswa', 'rasio_dosen_mahasiswa', 'nilai_rasio']
print(df[cols_tampil].head(5))

print('\n[TABEL 4] Ringkasan Data:')
summary = pd.DataFrame({
    'Keterangan': ['Total Baris (semua periode)', 'Jumlah Prodi Unik', 'Jumlah Periode',
                   'Periode Pertama', 'Periode Terakhir',
                   'Rasio Terendah (1:x)', 'Rasio Tertinggi (1:x)',
                   'Baris dengan nilai NaN (prodi 0 mahasiswa)'],
    'Nilai': [
        len(df),
        df['nama_program_studi'].nunique(),
        df['tahun_pelaporan'].nunique(),
        df['tahun_pelaporan'].min(),
        df['tahun_pelaporan'].max(),
        f"1:{df['nilai_rasio'].min():.2f}",
        f"1:{df['nilai_rasio'].max():.2f}",
        int(df['nilai_rasio'].isna().sum())
    ]
})
print(summary)

[TABEL 3] Sample Data Setelah Transformasi (5 baris pertama):
      tahun_pelaporan semester tahun nama_program_studi jenjang  total_dosen  \
15214     Ganjil 2023   Ganjil  2023         Agribisnis      S1           18   
15215     Ganjil 2023   Ganjil  2023         Agribisnis      S2            5   
15216     Ganjil 2023   Ganjil  2023      Agroteknologi      S1           15   
15217     Ganjil 2023   Ganjil  2023      Agroteknologi      S2            5   
15218     Ganjil 2023   Ganjil  2023          Akuntansi      S1           19   

       jumlah_mahasiswa rasio_dosen_mahasiswa  nilai_rasio  
15214               667               1:12.83        12.83  
15215                22                 1:2.2         2.20  
15216               568               1:13.21        13.21  
15217                36                   1:4         4.00  
15218              1176                1:33.6        33.60  

[TABEL 4] Ringkasan Data:
                                   Keterangan        Nilai
0    

---
## 4. LOAD — Pembentukan Star Schema

Struktur Star Schema:
```
                  ┌─────────────────┐
                  │   Dim_Waktu     │
                  │ PK: id_waktu    │
                  └────────┬────────┘
                           │
  ┌──────────────┐    ┌────┴──────────────────────────┐    ┌────────────────┐
  │Dim_Universitas│──│  Fact_Kapasitas_Pendidikan    │──  │   Dim_Prodi    │
  └───────────────┘  │  jumlah_dosen, jumlah_mahasiswa│    └────────────────┘
                     │  nilai_rasio                   │
                     └────────────────────────────────┘
```

Seluruh tabel disimpan ke Google Drive: `My Drive/College/Star_Schema/`

In [8]:
# Dim_Waktu
dim_waktu = (
    df[['tahun_pelaporan', 'semester', 'tahun']]
    .drop_duplicates()
    .sort_values('tahun_pelaporan')
    .reset_index(drop=True)
)
dim_waktu.insert(0, 'id_waktu', dim_waktu.index + 1)
dim_waktu['tahun'] = dim_waktu['tahun'].astype(int)

print('[Dim_Waktu]')
print(dim_waktu)

[Dim_Waktu]
   id_waktu tahun_pelaporan semester  tahun
0         1     Ganjil 2023   Ganjil   2023
1         2     Ganjil 2024   Ganjil   2024
2         3     Ganjil 2025   Ganjil   2025
3         4      Genap 2023    Genap   2023
4         5      Genap 2024    Genap   2024


In [9]:
# Dim_Universitas — data resmi Unsil
dim_univ = pd.DataFrame([{
    'id_universitas'      : '002008',
    'nama_universitas'    : 'Universitas Siliwangi',
    'kota'               : 'Kota Tasikmalaya',
    'provinsi'           : 'Prov. Jawa Barat',
    'status_pt'          : 'PTN',
    'akreditasi_institusi': 'Unggul'
}])

print('[Dim_Universitas]')
print(dim_univ)

[Dim_Universitas]
  id_universitas       nama_universitas              kota          provinsi  \
0         002008  Universitas Siliwangi  Kota Tasikmalaya  Prov. Jawa Barat   

  status_pt akreditasi_institusi  
0       PTN               Unggul  


In [10]:
# Dim_Prodi (gunakan data periode terbaru sebagai referensi atribut)
latest_period = df['tahun_pelaporan'].max()
dim_prodi = (
    df[df['tahun_pelaporan'] == latest_period]
    [['kode_prodi', 'nama_program_studi', 'jenjang', 'status_prodi', 'akreditasi_prodi']]
    .drop_duplicates(subset=['kode_prodi'])
    .sort_values('nama_program_studi')
    .reset_index(drop=True)
    .rename(columns={'kode_prodi': 'id_prodi'})
)

print(f'[Dim_Prodi] — {len(dim_prodi)} program studi (referensi periode: {latest_period})')
print(dim_prodi)

[Dim_Prodi] — 40 program studi (referensi periode: Genap 2024)
    id_prodi                          nama_program_studi  jenjang  \
0      54201                                  Agribisnis       S1   
1      54101                                  Agribisnis       S2   
2      54211                               Agroteknologi       S1   
3      54111                               Agroteknologi       S2   
4      62201                                   Akuntansi       S1   
5      60201                         Ekonomi Pembangunan       S1   
6      60202                            Ekonomi Syari'ah       S1   
7      13211                                        Gizi       S1   
8      61001                              Ilmu Manajemen       S3   
9      54001                              Ilmu Pertanian       S3   
10     67201                                Ilmu Politik       S1   
11     55201                                 Informatika       S1   
12     13201                        Kese

In [11]:
# Fact_Kapasitas_Pendidikan
fact = df.merge(dim_waktu[['id_waktu', 'tahun_pelaporan']], on='tahun_pelaporan', how='left')

fact_table = fact[[
    'kode_pt', 'kode_prodi', 'id_waktu',
    'jumlah_dosen_penghitung_rasio', 'dosen_tetap', 'dosen_tidak_tetap', 'total_dosen',
    'jumlah_mahasiswa', 'rasio_dosen_mahasiswa', 'nilai_rasio'
]].rename(columns={
    'kode_pt'   : 'id_universitas',
    'kode_prodi': 'id_prodi'
})

fact_table = fact_table.dropna(subset=['id_universitas', 'id_prodi']).reset_index(drop=True)

print(f'[Fact_Kapasitas_Pendidikan] — {len(fact_table)} baris')
print(f'Cakupan: {fact_table["id_waktu"].nunique()} periode | {fact_table["id_prodi"].nunique()} prodi unik')
print(fact_table.head(10))

[Fact_Kapasitas_Pendidikan] — 200 baris
Cakupan: 5 periode | 40 prodi unik
  id_universitas  id_prodi  id_waktu  jumlah_dosen_penghitung_rasio  \
0         002008     54201         1                             52   
1         002008     54101         1                             10   
2         002008     54211         1                             43   
3         002008     54111         1                              9   
4         002008     62201         1                             35   
5         002008     60201         1                             34   
6         002008     60202         1                             21   
7         002008     13211         1                             20   
8         002008     61001         1                              0   
9         002008     54001         1                              0   

   dosen_tetap  dosen_tidak_tetap  total_dosen  jumlah_mahasiswa  \
0           18                  0           18               667   
1      

---
## 5. Simpan Star Schema ke Google Drive

In [12]:
# Simpan 4 tabel star schema ke Google Drive/College/Star_Schema/
dim_waktu.to_csv(os.path.join(PATH_OUT_SCHEMA, 'Dim_Waktu.csv'), index=False)
dim_univ.to_csv(os.path.join(PATH_OUT_SCHEMA, 'Dim_Universitas.csv'), index=False)
dim_prodi.to_csv(os.path.join(PATH_OUT_SCHEMA, 'Dim_Prodi.csv'), index=False)
fact_table.to_csv(os.path.join(PATH_OUT_SCHEMA, 'Fact_Kapasitas_Pendidikan.csv'), index=False)

print('✅ Star Schema tersimpan di Google Drive:', PATH_OUT_SCHEMA)
for f in ['Dim_Waktu.csv', 'Dim_Universitas.csv', 'Dim_Prodi.csv', 'Fact_Kapasitas_Pendidikan.csv']:
    path = os.path.join(PATH_OUT_SCHEMA, f)
    rows = pd.read_csv(path).shape[0]
    print(f'   {f}: {rows} baris')

✅ Star Schema tersimpan di Google Drive:

 d:/Code/Tugas Akhir/Data/Star_Schema/


   Dim_Waktu.csv: 5 baris
   Dim_Universitas.csv: 1 baris


   Dim_Prodi.csv: 40 baris
   Fact_Kapasitas_Pendidikan.csv: 200 baris


---
## 6. Buat Flat Table (master_looker_unsil.csv)

Flat table ini menggabungkan semua dimensi dan fakta menjadi satu tabel untuk Google Looker Studio.
Disimpan ke: `My Drive/College/Processed/master_looker_unsil.csv`

In [13]:
master = df[[
    'tahun_pelaporan', 'semester', 'tahun',
    'nama_program_studi', 'jenjang', 'status_prodi', 'akreditasi_prodi',
    'nama_universitas',
    'jumlah_mahasiswa', 'jumlah_dosen_penghitung_rasio', 'dosen_tetap',
    'dosen_tidak_tetap', 'total_dosen',
    'rasio_dosen_mahasiswa', 'nilai_rasio'
]].copy()

master['kota']     = 'Kota Tasikmalaya'
master['provinsi'] = 'Prov. Jawa Barat'
master['kode_pt']  = '002008'

master = master.sort_values(['tahun_pelaporan', 'nama_program_studi']).reset_index(drop=True)
master.to_csv(PATH_OUT_MASTER, index=False)

print(f'✅ master_looker_unsil.csv tersimpan ke Google Drive: {len(master)} baris')
print(f'   Prodi : {master["nama_program_studi"].nunique()} prodi unik')
print(f'   Periode: {sorted(master["tahun_pelaporan"].unique())}')
print(master.head())

✅ master_looker_unsil.csv tersimpan ke Google Drive: 200 baris
   Prodi : 34 prodi unik
   Periode: ['Ganjil 2023', 'Ganjil 2024', 'Ganjil 2025', 'Genap 2023', 'Genap 2024']
  tahun_pelaporan semester tahun nama_program_studi jenjang status_prodi  \
0     Ganjil 2023   Ganjil  2023         Agribisnis      S1        Aktif   
1     Ganjil 2023   Ganjil  2023         Agribisnis      S2        Aktif   
2     Ganjil 2023   Ganjil  2023      Agroteknologi      S1        Aktif   
3     Ganjil 2023   Ganjil  2023      Agroteknologi      S2        Aktif   
4     Ganjil 2023   Ganjil  2023          Akuntansi      S1        Aktif   

  akreditasi_prodi       nama_universitas  jumlah_mahasiswa  \
0      Baik Sekali  Universitas Siliwangi               667   
1      Baik Sekali  Universitas Siliwangi                22   
2      Baik Sekali  Universitas Siliwangi               568   
3             Baik  Universitas Siliwangi                36   
4      Baik Sekali  Universitas Siliwangi             

---
## ✅ ETL Selesai

| Output | Lokasi di Google Drive | Keterangan |
|--------|------------------------|------------|
| Dim_Waktu | `College/Star_Schema/Dim_Waktu.csv` | 5 periode |
| Dim_Universitas | `College/Star_Schema/Dim_Universitas.csv` | 1 PT (Unsil) |
| Dim_Prodi | `College/Star_Schema/Dim_Prodi.csv` | 35 prodi aktif |
| Fact Table | `College/Star_Schema/Fact_Kapasitas_Pendidikan.csv` | 201 observasi |
| Flat Table | `College/Processed/master_looker_unsil.csv` | Untuk Looker Studio |

**Langkah berikutnya:**
1. Upload `master_looker_unsil.csv` ke Google Sheets
2. Sambungkan Google Sheets ke Google Looker Studio sebagai Data Source
3. Jalankan notebook `Dashboard_Visualisasi.ipynb` untuk visualisasi Python